# Templat Notebook Mini-Project — PIF304 Pengantar Data Sains
**Artefak A6 (Implementasi mini-project, bobot 20%)** · dipakai mulai Pertemuan 9, dinilai pada Pertemuan 12, disempurnakan sampai Pertemuan 16

Notebook ini adalah **kerangka**, bukan jawaban. Urutan bagiannya mengikuti siklus kerja Data Sains dan rubrik A6:

| Bagian | Isi | Kriteria rubrik A6 |
|---|---|---|
| 1 | Masalah dan tugas analitik | Problem framing (15%) |
| 2–4 | Data, mutu data, EDA terarah | Persiapan/EDA (25%) |
| 5–7 | Fitur, pembagian data, baseline, model awal | Baseline/pemodelan awal (20%) |
| 8 | Evaluasi dan pemeriksaan error | Evaluasi (20%) |
| 9 | Log implementasi, kontribusi, versi pustaka | Reproducibility (10%) |
| 10 | Batas interpretasi, etika, keterbatasan | Etika/keterbatasan (10%) |

**Aturan pakai**
1. Simpan salinan ke Drive tim: *File → Save a copy in Drive*. Beri nama `MiniProject_PDS_TimX.ipynb`.
2. Semua bagian bertanda **TODO** wajib diisi. Sel kode boleh ditambah, diubah, atau dipecah.
3. Sebelum dikumpulkan: *Runtime → Restart and run all*. Notebook harus berjalan **dari atas sampai bawah tanpa error** dengan file data tim.
4. Setiap keputusan (membuang baris, memilih fitur, memilih ambang) dicatat dengan `catat(...)` agar muncul di log implementasi.

## 0. Identitas Tim
**TODO:** lengkapi tabel dan sel kode di bawah.

| Peran | Nama | NIM |
|---|---|---|
| Koordinator | … | … |
| Pengelola data | … | … |
| Analis EDA dan visualisasi | … | … |
| Pemodel dan evaluasi | … | … |
| Dokumentasi dan etika | … | … |

In [ ]:
TIM = ""            # TODO: nama/nomor tim
JUDUL = ""          # TODO: judul mini-project
ANGGOTA = {         # TODO: NIM -> nama
    # "2410001": "Nama Anggota 1",
}
print("Tim:", TIM, "| Judul:", JUDUL, "| Jumlah anggota:", len(ANGGOTA))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import accuracy_score, precision_score, recall_score, mean_absolute_error, confusion_matrix
import sys, sklearn, matplotlib
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
sns.set_theme(style="whitegrid")
# Palet grafik (sudah diuji aman untuk buta warna): satu seri = TEAL; pembanding = ORANGE, VIOLET; latar = GREY
NAVY, TEAL, ORANGE, VIOLET, GREY = "#13233F", "#1293A6", "#E4683A", "#6C63C9", "#B8BDC4"
PALET = [TEAL, ORANGE, VIOLET]

In [ ]:
# Log implementasi: setiap keputusan penting dicatat di sini (dipakai di Bagian 9)
log = []
def catat(tahap, keputusan, alasan, oleh=""):
    """tahap: Data / Pembersihan / EDA / Fitur / Pembagian / Baseline / Model / Evaluasi"""
    log.append({"no": len(log) + 1, "tahap": tahap, "keputusan": keputusan, "alasan": alasan, "oleh": oleh})

def nilai_klasifikasi(nama, y_benar, y_tebak):
    return {"model": nama,
            "accuracy": round(accuracy_score(y_benar, y_tebak), 3),
            "precision": round(precision_score(y_benar, y_tebak, zero_division=0), 3),
            "recall": round(recall_score(y_benar, y_tebak, zero_division=0), 3)}

def nilai_regresi(nama, y_benar, y_tebak):
    return {"model": nama, "MAE": round(mean_absolute_error(y_benar, y_tebak), 3)}
print("Fungsi bantu siap: catat(), nilai_klasifikasi(), nilai_regresi()")

## 1. Masalah dan Tugas Analitik
**TODO:** tulis ulang dari proposal tim (A5). Maksimal 200 kata.

**Problem statement (lima komponen)**
- **Pemangku kepentingan:** …
- **Masalah dan bukti awal:** …
- **Mengapa penting:** …
- **Keputusan yang didukung:** …
- **Kriteria keberhasilan (terukur):** …

**Pertanyaan analitik**
1. (deskriptif/diagnostik) …
2. (prediktif) …

**Rumusan tugas prediksi**

| Unsur | Isi |
|---|---|
| Target | … |
| Jenis tugas | regresi / klasifikasi |
| Unit analisis | satu baris = … |
| Waktu prediksi | prediksi dibuat pada saat … (hanya informasi yang tersedia sebelum saat itu boleh menjadi fitur) |
| Pemakai hasil | … |
| Biaya salah tebak | salah tebak jenis … lebih mahal karena … |

## 2. Data
**TODO:** lengkapi keterangan sumber data.

| Unsur | Isi |
|---|---|
| Nama dataset | … |
| Sumber / tautan | … |
| Pemilik dan izin pakai (lisensi) | … |
| Waktu dan cara pengumpulan | … |
| Tanggal diunduh | … |
| Catatan privasi | tidak memuat nama, NIM, NIK, nomor telepon, alamat lengkap, atau gabungan kolom yang dapat mengenali seseorang |

### Unggah File Data Tim
Jalankan sel di bawah, klik **Choose Files**, lalu pilih file data tim (`.csv` atau `.xlsx`) — file yang **sama persis** dengan yang dikumpulkan bersama notebook. Notebook **tidak dapat dilanjutkan** sebelum file berhasil diunggah.

In [ ]:
NAMA_FILE_LOKAL = "data_tim.csv"   # hanya dipakai bila notebook dijalankan di luar Google Colab
try:
    from google.colab import files
    print("Silakan pilih file data tim (.csv atau .xlsx)")
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise ValueError("Belum ada file yang diunggah. Jalankan ulang sel ini dan pilih file-nya.")
    nama_file = list(uploaded.keys())[0]
except ImportError:
    nama_file = NAMA_FILE_LOKAL

if nama_file.lower().endswith((".xlsx", ".xls")):
    df_mentah = pd.read_excel(nama_file)
else:
    df_mentah = pd.read_csv(nama_file)   # bila pemisahnya titik koma: pd.read_csv(nama_file, sep=";")
print("File:", nama_file, "| ukuran:", df_mentah.shape)
catat("Data", f"Memuat {nama_file} ({df_mentah.shape[0]} baris, {df_mentah.shape[1]} kolom)", "Data mentah tidak diubah; semua perubahan dilakukan pada salinan")
df_mentah.head()

In [ ]:
# Profil singkat setiap kolom -> bahan data dictionary
profil = pd.DataFrame({
    "tipe": df_mentah.dtypes.astype(str),
    "kosong": df_mentah.isna().sum(),
    "kosong_%": (df_mentah.isna().mean() * 100).round(1),
    "nilai_unik": df_mentah.nunique(),
    "contoh": df_mentah.apply(lambda s: ", ".join(map(str, s.dropna().unique()[:3]))),
})
profil

### Data Dictionary
**TODO:** isi untuk setiap kolom yang dipakai (minimal target dan seluruh fitur).

| Kolom | Definisi | Skala ukur | Satuan / kategori | Nilai valid | Tersedia sebelum waktu prediksi? | Catatan mutu |
|---|---|---|---|---|---|---|
| … | … | nominal / ordinal / interval / rasio | … | … | ya / tidak | … |

## 3. Konfigurasi Analisis
Isi sel berikut **setelah** melihat nama kolom di atas. Seluruh bagian sesudahnya memakai konfigurasi ini, sehingga satu perubahan di sini berlaku ke semua langkah.

- `KOLOM_BOCOR`: kolom yang baru diketahui **setelah** waktu prediksi atau merupakan turunan langsung dari target. Kolom ini **tidak boleh** menjadi fitur.
- Untuk klasifikasi dengan lebih dari dua kelas, pilih satu kelas sebagai `KELAS_POSITIF` (satu-lawan-lainnya).

In [ ]:
JENIS_TUGAS = ""        # TODO: "klasifikasi" atau "regresi"
TARGET = ""             # TODO: nama kolom target
KELAS_POSITIF = None    # TODO (hanya klasifikasi): nilai target yang ingin dideteksi, mis. "Ya"
KOLOM_ID = []           # TODO: kolom pengenal baris (tidak dipakai sebagai fitur)
KOLOM_BOCOR = []        # TODO: kolom yang membocorkan target (tidak dipakai sebagai fitur)
FITUR_NUMERIK = []      # TODO: fitur berskala interval/rasio
FITUR_KATEGORI = []     # TODO: fitur berskala nominal/ordinal
RANDOM_STATE = 42       # jangan diubah-ubah untuk mencari hasil terbaik

def periksa_konfigurasi():
    kurang = []
    if JENIS_TUGAS not in ("klasifikasi", "regresi"): kurang.append('JENIS_TUGAS harus "klasifikasi" atau "regresi"')
    if TARGET not in df_mentah.columns: kurang.append("TARGET harus salah satu nama kolom data")
    if len(FITUR_NUMERIK) + len(FITUR_KATEGORI) == 0: kurang.append("isi minimal satu fitur")
    tak_ada = [k for k in FITUR_NUMERIK + FITUR_KATEGORI + KOLOM_ID + KOLOM_BOCOR if k not in df_mentah.columns]
    if tak_ada: kurang.append(f"kolom tidak ditemukan: {tak_ada}")
    terlarang = [k for k in FITUR_NUMERIK + FITUR_KATEGORI if k in KOLOM_ID + KOLOM_BOCOR + [TARGET]]
    if terlarang: kurang.append(f"kolom ID/bocor/target tidak boleh menjadi fitur: {terlarang}")
    if JENIS_TUGAS == "klasifikasi" and TARGET in df_mentah.columns and KELAS_POSITIF not in set(df_mentah[TARGET].dropna().unique()):
        kurang.append("KELAS_POSITIF harus salah satu nilai pada kolom target")
    if kurang:
        raise ValueError("Konfigurasi belum lengkap:\n- " + "\n- ".join(kurang))
    print("Konfigurasi lengkap:", JENIS_TUGAS, "| target:", TARGET, "| fitur:", len(FITUR_NUMERIK), "numerik +", len(FITUR_KATEGORI), "kategori")

periksa_konfigurasi()
KLASIFIKASI = JENIS_TUGAS == "klasifikasi"
catat("Fitur", f"Target = {TARGET}; kolom tidak dipakai: ID {KOLOM_ID}, bocor {KOLOM_BOCOR}", "TODO: tulis alasan tim")

## 4. Mutu Data dan Pembersihan
Pola Pertemuan 5: **temukan → putuskan → catat**. Data mentah (`df_mentah`) tidak pernah diubah; semua pembersihan dilakukan pada salinan `df`.

In [ ]:
df = df_mentah.copy()

# (a) duplikat
n_dup = int(df.duplicated().sum())
print("Baris duplikat persis:", n_dup)
if n_dup > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    catat("Pembersihan", f"Menghapus {n_dup} baris duplikat persis", "Satu unit analisis hanya boleh muncul sekali")

# (b) baris tanpa target tidak dapat dipakai untuk melatih maupun menguji
n_tanpa = int(df[TARGET].isna().sum())
if n_tanpa > 0:
    df = df[df[TARGET].notna()].reset_index(drop=True)
    catat("Pembersihan", f"Mengeluarkan {n_tanpa} baris tanpa nilai target", "Target tidak boleh diisi dengan tebakan")

# (c) fitur numerik harus benar-benar bertipe angka
for k in FITUR_NUMERIK:
    sebelum = int(df[k].isna().sum())
    df[k] = pd.to_numeric(df[k], errors="coerce")
    gagal = int(df[k].isna().sum()) - sebelum
    if gagal > 0:
        catat("Pembersihan", f"{k}: {gagal} nilai bukan angka dijadikan kosong", "Teks pada kolom numerik tidak dapat dihitung")

print("Ukuran setelah langkah (a)-(c):", df.shape)
df[FITUR_NUMERIK].describe().T if FITUR_NUMERIK else None

In [ ]:
# TODO (wajib): pembersihan khusus dataset tim. Contoh pola (hapus tanda # dan sesuaikan):
# df["kota"] = df["kota"].str.strip().str.title()                       # ejaan kategori tidak konsisten
# df.loc[df["usia"] > 100, "usia"] = np.nan                             # nilai mustahil menurut data dictionary
# catat("Pembersihan", "usia > 100 dijadikan kosong (3 baris)", "Di luar rentang valid 15-100", oleh="NIM")

print("Nilai kosong per kolom yang dipakai:")
df[[TARGET] + FITUR_NUMERIK + FITUR_KATEGORI].isna().sum()

## 5. EDA Terarah
Pola Pertemuan 9: **pertanyaan → visual → statistik → temuan / dugaan / keterbatasan**. Mulai dari target, lalu hubungan setiap fitur dengan target.

In [ ]:
# 5.1 Sebaran target
fig, ax = plt.subplots(figsize=(6.4, 3.6))
if KLASIFIKASI:
    hitung = df[TARGET].value_counts()
    ax.bar(hitung.index.astype(str), hitung.values, color=TEAL)
    for i, v in enumerate(hitung.values): ax.text(i, v, f"{v} ({v / len(df):.0%})", ha="center", va="bottom")
    ax.set_ylabel("jumlah baris")
else:
    ax.hist(df[TARGET], bins=20, color=TEAL, edgecolor="white")
    ax.axvline(df[TARGET].median(), color=NAVY, linestyle="--"); ax.set_ylabel("jumlah baris")
ax.set_xlabel(TARGET)
ax.set_title("TODO: ganti dengan kalimat temuan tentang target", loc="left")
plt.tight_layout(); plt.show()

if KLASIFIKASI:
    p_pos = (df[TARGET] == KELAS_POSITIF).mean()
    print(f"Proporsi kelas positif ({KELAS_POSITIF}): {p_pos:.1%} -> menebak 'selalu kelas terbanyak' saja sudah benar sekitar {max(p_pos, 1 - p_pos):.0%}")
else:
    print(df[TARGET].describe().round(2))

In [ ]:
# 5.2 Fitur numerik vs target
y_angka = (df[TARGET] == KELAS_POSITIF).astype(int) if KLASIFIKASI else df[TARGET]
if FITUR_NUMERIK:
    kor = df[FITUR_NUMERIK].corrwith(y_angka, method="spearman").sort_values()
    fig, ax = plt.subplots(figsize=(6.4, 0.45 * len(kor) + 1.4))
    ax.barh(kor.index, kor.values, color=TEAL); ax.axvline(0, color=NAVY, linewidth=1)
    ax.set_xlabel("korelasi Spearman dengan target"); ax.set_xlim(-1, 1)
    ax.set_title("TODO: fitur mana yang paling berhubungan dengan target?", loc="left")
    plt.tight_layout(); plt.show()
    print(kor.round(2).to_string())
    print("\nIngat: korelasi bukan bukti sebab-akibat, dan kolom dengan korelasi mendekati +1 atau -1 patut dicurigai sebagai kebocoran.")

In [ ]:
# 5.3 Fitur kategori vs target
for k in FITUR_KATEGORI:
    ringkas = df.groupby(k, dropna=False)[TARGET].agg(
        n="size", nilai=(lambda s: (s == KELAS_POSITIF).mean()) if KLASIFIKASI else "median")
    ringkas = ringkas.rename(columns={"nilai": f"proporsi {KELAS_POSITIF}" if KLASIFIKASI else f"median {TARGET}"}).round(3)
    print(f"\n{k}"); print(ringkas.sort_values("n", ascending=False).head(10).to_string())
print("\nPerhatikan kolom n: kelompok dengan n kecil tidak layak dijadikan dasar kesimpulan.")

In [ ]:
# TODO (wajib): minimal DUA grafik EDA lain yang menjawab pertanyaan analitik tim.
# Setiap grafik: judul berupa temuan, sumbu berlabel, dan satu kalimat interpretasi di sel teks sesudahnya.

### Ringkasan Temuan EDA
**TODO:** minimal empat baris. Bedakan dengan tegas apa yang *terlihat di data* dan apa yang masih *dugaan*.

| No | Temuan (apa yang terlihat, dengan angka) | Dugaan (mengapa) | Keterbatasan | Dampak pada pemodelan |
|---|---|---|---|---|
| 1 | … | … | … | … |
| 2 | … | … | … | … |

## 6. Fitur dan Pembagian Data
Pola Pertemuan 10. Urutan yang **tidak boleh dibalik**: bagi data dulu, baru hitung apa pun yang "belajar" dari data (median untuk imputasi) — dan hanya dari data latih.

In [ ]:
X = df[FITUR_NUMERIK].copy()
if FITUR_KATEGORI:
    kategori = df[FITUR_KATEGORI].astype("string").fillna("(kosong)")
    X = pd.concat([X, pd.get_dummies(kategori, drop_first=True, dtype=int)], axis=1)

# TODO (opsional): fitur rekayasa sederhana, mis. selisih atau rasio dua kolom. Catat dengan catat("Fitur", ...).

y = (df[TARGET] == KELAS_POSITIF).astype(int) if KLASIFIKASI else df[TARGET].astype(float)

X_latih, X_uji, y_latih, y_uji = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y if KLASIFIKASI else None)

median_latih = X_latih.median()                 # dihitung dari data latih saja
X_latih, X_uji = X_latih.fillna(median_latih), X_uji.fillna(median_latih)

catat("Pembagian", f"80/20, random_state={RANDOM_STATE}" + (", stratify=target" if KLASIFIKASI else ""), "Data uji disimpan sebagai ujian akhir; proporsi kelas dijaga" if KLASIFIKASI else "Data uji disimpan sebagai ujian akhir")
catat("Fitur", f"{X.shape[1]} kolom fitur; nilai kosong diisi median data latih", "Mencegah informasi data uji bocor ke proses persiapan")
print("Latih:", X_latih.shape, "| Uji:", X_uji.shape)
if KLASIFIKASI: print(f"Proporsi positif - latih: {y_latih.mean():.1%} | uji: {y_uji.mean():.1%}")

## 7. Baseline dan Model Awal
Pola Pertemuan 10–11. Model baru layak disebut berguna bila **mengalahkan baseline pada data uji**.

In [ ]:
hasil = []

# Baseline 1: tanpa informasi apa pun
if KLASIFIKASI:
    dasar = DummyClassifier(strategy="most_frequent").fit(X_latih, y_latih)
    hasil.append(nilai_klasifikasi("Baseline: selalu kelas terbanyak", y_uji, dasar.predict(X_uji)))
else:
    dasar = DummyRegressor(strategy="mean").fit(X_latih, y_latih)
    hasil.append(nilai_regresi("Baseline: selalu rata-rata data latih", y_uji, dasar.predict(X_uji)))

# Baseline 2 - TODO (wajib): aturan sederhana dari temuan EDA tim. Contoh:
#   klasifikasi: pred_aturan = (X_uji["nama_kolom"] < 2.75).astype(int)
#   regresi    : pred_aturan = X_uji["nama_kolom"]
pred_aturan = None
if pred_aturan is not None:
    hasil.append((nilai_klasifikasi if KLASIFIKASI else nilai_regresi)("Baseline: aturan sederhana tim", y_uji, pred_aturan))
    catat("Baseline", "Aturan sederhana: TODO tulis aturannya", "Diturunkan dari temuan EDA nomor ...")
else:
    print("PERINGATAN: baseline aturan sederhana belum diisi (wajib untuk A6).")
pd.DataFrame(hasil)

In [ ]:
# Model awal: satu model sederhana yang dapat dijelaskan
if KLASIFIKASI:
    model = LogisticRegression(max_iter=5000).fit(X_latih, y_latih)      # alternatif: DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
    nama_model = "Model awal: regresi logistik"
    hasil.append(nilai_klasifikasi(nama_model, y_uji, model.predict(X_uji)))
else:
    model = LinearRegression().fit(X_latih, y_latih)                     # alternatif: DecisionTreeRegressor(max_depth=3, random_state=RANDOM_STATE)
    nama_model = "Model awal: regresi linear"
    hasil.append(nilai_regresi(nama_model, y_uji, model.predict(X_uji)))
catat("Model", nama_model, "TODO: mengapa model ini sesuai dengan masalah tim?")

perbandingan = pd.DataFrame(hasil)
perbandingan

## 8. Evaluasi dan Pemeriksaan Error
Pola Pertemuan 11 dan 13: angka metrik baru bermakna setelah (1) dibandingkan dengan baseline, (2) diperiksa di mana model salah, dan (3) diuji kestabilannya.

In [ ]:
pred_uji, pred_latih = model.predict(X_uji), model.predict(X_latih)

if KLASIFIKASI:
    tn, fp, fn, tp = confusion_matrix(y_uji, pred_uji, labels=[0, 1]).ravel()
    print(f"Confusion matrix data uji (positif = {KELAS_POSITIF})")
    print(pd.DataFrame([[tn, fp], [fn, tp]], index=["sebenarnya: negatif", "sebenarnya: positif"], columns=["ditebak: negatif", "ditebak: positif"]).to_string())
    print(f"\nSalah alarm (FP): {fp} | Terlewat (FN): {fn}  -> mana yang lebih mahal bagi pemakai hasil?")
    print(f"Accuracy data latih {accuracy_score(y_latih, pred_latih):.3f} vs data uji {accuracy_score(y_uji, pred_uji):.3f}  (selisih besar = tanda overfitting)")
else:
    galat = (y_uji - pred_uji)
    print(f"MAE data latih {mean_absolute_error(y_latih, pred_latih):.3f} vs data uji {mean_absolute_error(y_uji, pred_uji):.3f}  (selisih besar = tanda overfitting)")
    fig, ax = plt.subplots(figsize=(5.2, 4.2))
    ax.scatter(y_uji, pred_uji, color=TEAL, alpha=0.6, s=22)
    batas = [min(y_uji.min(), pred_uji.min()), max(y_uji.max(), pred_uji.max())]
    ax.plot(batas, batas, color=NAVY, linestyle="--", linewidth=1)
    ax.set_xlabel(f"{TARGET} sebenarnya"); ax.set_ylabel("tebakan model")
    ax.set_title("TODO: seberapa dekat tebakan dengan kenyataan?", loc="left")
    plt.tight_layout(); plt.show()

In [ ]:
# Pemeriksaan error: baris data uji yang paling salah ditebak
periksa = df.loc[X_uji.index, [TARGET] + FITUR_NUMERIK + FITUR_KATEGORI].copy()
if KLASIFIKASI:
    periksa["peluang_positif"] = model.predict_proba(X_uji)[:, 1].round(2)
    periksa["benar"] = (pred_uji == y_uji.values)
    tampil = periksa[~periksa["benar"]].sort_values("peluang_positif")
else:
    periksa["tebakan"] = pred_uji.round(2)
    periksa["galat_mutlak"] = (y_uji - pred_uji).abs().round(2)
    tampil = periksa.sort_values("galat_mutlak", ascending=False)
print("Jumlah baris data uji:", len(periksa))
tampil.head(8)

In [ ]:
# Uji kestabilan: apakah hasil berubah banyak bila pembagian data berbeda?
ulang = []
for seed in [1, 7, 21, 42, 99]:
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y if KLASIFIKASI else None)
    med = Xa.median(); Xa, Xb = Xa.fillna(med), Xb.fillna(med)
    if KLASIFIKASI:
        m = LogisticRegression(max_iter=5000).fit(Xa, ya) if isinstance(model, LogisticRegression) else DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE).fit(Xa, ya)
        d = DummyClassifier(strategy="most_frequent").fit(Xa, ya)
        ulang.append({"seed": seed, "baseline_accuracy": round(accuracy_score(yb, d.predict(Xb)), 3), "model_accuracy": round(accuracy_score(yb, m.predict(Xb)), 3), "model_recall": round(recall_score(yb, m.predict(Xb), zero_division=0), 3)})
    else:
        m = LinearRegression().fit(Xa, ya) if isinstance(model, LinearRegression) else DecisionTreeRegressor(max_depth=3, random_state=RANDOM_STATE).fit(Xa, ya)
        d = DummyRegressor().fit(Xa, ya)
        ulang.append({"seed": seed, "baseline_MAE": round(mean_absolute_error(yb, d.predict(Xb)), 3), "model_MAE": round(mean_absolute_error(yb, m.predict(Xb)), 3)})
kestabilan = pd.DataFrame(ulang)
print(kestabilan.to_string(index=False))
print("\nRentang antar-pembagian:"); print((kestabilan.drop(columns="seed").max() - kestabilan.drop(columns="seed").min()).round(3).to_string())
print("\nBila selisih model vs baseline lebih kecil daripada rentang ini, jangan klaim model 'lebih baik'.")

### Tiga Kalimat Wajib
**TODO:** tulis dengan angka dari tabel di atas.

1. **Dibanding baseline:** model … menghasilkan … pada data uji, sedangkan baseline terbaik … ; selisihnya … dan (stabil / tidak stabil) pada lima pembagian data.
2. **Arti bagi pemakai:** dari … kasus pada data uji, model salah pada … kasus; jenis salah yang paling merugikan adalah … karena …
3. **Batas klaim:** hasil ini berlaku untuk … dan belum tentu berlaku untuk … ; model ini **tidak** menunjukkan bahwa … menyebabkan …

### Daftar Perbaikan (Pertemuan 13)
**TODO:** minimal tiga butir, urut dari yang paling berdampak.

| No | Kelemahan yang ditemukan | Bukti | Rencana perbaikan | Status |
|---|---|---|---|---|
| 1 | … | … | … | belum / sudah |

## 9. Reproducibility: Log Implementasi, Kontribusi, dan Versi Pustaka
Log di bawah tersusun otomatis dari setiap pemanggilan `catat(...)`. **TODO:** lengkapi kolom `oleh` saat mencatat, lalu isi tabel kontribusi.

In [ ]:
log_df = pd.DataFrame(log)
log_df.to_csv("log_implementasi_tim.csv", index=False)
perbandingan.to_csv("perbandingan_model_tim.csv", index=False)
print("Tersimpan: log_implementasi_tim.csv, perbandingan_model_tim.csv (unduh dari panel Files dan sertakan saat pengumpulan)")
log_df

In [ ]:
# TODO: kontribusi setiap anggota (ringkas, dapat diperiksa)
kontribusi = pd.DataFrame([
    # {"NIM": "2410001", "nama": "Nama", "peran": "Pengelola data", "kontribusi": "Membersihkan kolom X dan Y; menulis data dictionary", "bagian_notebook": "2, 4"},
])
print("versi | python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__, "| matplotlib", matplotlib.__version__)
kontribusi

## 10. Batas Interpretasi, Etika, dan Keterbatasan
**TODO:** jawab dengan jujur dan spesifik untuk dataset tim (bukan pernyataan umum).

| Aspek | Jawaban tim |
|---|---|
| Siapa yang **tidak terwakili** dalam data ini? | … |
| Kolom sensitif (jenis kelamin, agama, suku, daerah, kondisi ekonomi, kesehatan) — dipakai atau tidak, dan mengapa? | … |
| Apa akibatnya bagi orang yang **salah ditebak** oleh model? | … |
| Klaim yang **tidak boleh** ditarik dari hasil ini | … |
| Keterbatasan data (ukuran, waktu, cara pengumpulan) | … |
| Keterbatasan metode | … |
| Bila dilanjutkan, data atau langkah apa yang paling dibutuhkan? | … |

## Pernyataan Penggunaan AI Generatif
**TODO:** isi apa adanya. Menulis "tidak memakai" padahal memakai adalah pelanggaran integritas akademik.

| Alat | Dipakai untuk bagian | Apa yang diubah/diperiksa tim dari keluarannya |
|---|---|---|
| … | … | … |

## Daftar Periksa Sebelum Mengumpulkan
- [ ] *Restart and run all* berjalan tanpa error dengan file data yang dikumpulkan
- [ ] Tidak ada tulisan TODO yang tersisa
- [ ] Setiap grafik berjudul temuan, bersumbu jelas, dan diikuti satu kalimat interpretasi
- [ ] Kolom bocor dan kolom ID tidak menjadi fitur
- [ ] Tabel perbandingan memuat minimal dua baseline dan satu model
- [ ] Tiga kalimat wajib memakai angka dari notebook ini
- [ ] Log implementasi dan tabel kontribusi terisi
- [ ] Data tidak memuat informasi yang dapat mengenali orang